In [84]:
import numpy as np
import pandas as pd
import yfinance as yfin
from fredapi import Fred

In [85]:
df = pd.read_parquet("sp500_prices.parquet")
df

,Date,Open,High,Low,Close,Volume,Ticker
0,2001-01-02,32.129270,32.129270,29.259251,30.340168,2261684,A
1,2001-01-02,0.222645,0.228257,0.217968,0.222645,452312000,AAPL
2,2001-01-02,11.261874,11.495887,11.232622,11.261874,5243919,ABT
3,2001-01-02,1.571624,1.571624,1.532003,1.532003,90000,ACGL
4,2001-01-02,29.005323,29.067432,22.235344,23.221338,16813400,ADBE
...,...,...,...,...,...,...,...
3747289,2026-05-19,152.009995,153.695007,150.669998,153.335007,650188,YUM
3747290,2026-05-19,85.250000,86.364998,84.290001,86.089996,923849,ZBH
3747291,2026-05-19,259.640015,259.845001,249.699997,250.520004,477407,ZBRA
3747292,2026-05-19,60.130001,60.470001,59.189999,60.310001,367297,ZION


In [86]:
df = df.set_index(["Date","Ticker"]).sort_index()
df

Open        High         Low       Close     Volume
Date       Ticker                                                           
2001-01-02 A        32.129270   32.129270   29.259251   30.340168    2261684
           AAPL      0.222645    0.228257    0.217968    0.222645  452312000
           ABT      11.261874   11.495887   11.232622   11.261874    5243919
           ACGL      1.571624    1.571624    1.532003    1.532003      90000
           ADBE     29.005323   29.067432   22.235344   23.221338   16813400
...                       ...         ...         ...         ...        ...
2026-05-19 YUM     152.009995  153.695007  150.669998  153.335007     650188
           ZBH      85.250000   86.364998   84.290001   86.089996     923849
           ZBRA    259.640015  259.845001  249.699997  250.520004     477407
           ZION     60.130001   60.470001   59.189999   60.310001     367297
           ZTS      79.849998   80.360001   77.593002   77.889999    3895872

[3747294 rows x 5 columns]

In [87]:
df.to_csv("sp500_prices.csv")

In [88]:
fred = Fred(api_key='e46be1fc28833cbace41a97c33bb6e3b')
data = fred.get_series('SP500')
data

2016-05-20    2052.32
2016-05-23    2048.04
2016-05-24    2076.06
2016-05-25    2090.54
2016-05-26    2090.10
               ...   
2026-05-12    7400.96
2026-05-13    7444.25
2026-05-14    7501.24
2026-05-15    7408.50
2026-05-18    7403.05
Length: 2607, dtype: float64

In [89]:
CHECK = {
    "jobless_claims":   "ICSA",          # weekly initial claims, SA
    "nonfarm_payrolls": "PAYEMS",        # payrolls level (thousands)
    "payrolls_mom":     "PAYEMS",        # MoM -> derive, see note above
    "ig_spread":        "BAMLC0A0CM",    # ICE BofA US Corporate OAS
    "hy_spread":        "BAMLH0A0HYM2",  # ICE BofA US High Yield OAS
    "usd_index":        "DTWEXBGS",      # Nominal Broad USD Index
}

for name, sid in CHECK.items():
    try:
        s = fred.get_series(sid)
        print(f"{name:18s} {sid:14s} OK    n={len(s):5d}  "
              f"{s.first_valid_index().date()} -> {s.last_valid_index().date()}")
    except Exception as e:
        print(f"{name:18s} {sid:14s} FAIL  {type(e).__name__}: {e}")

jobless_claims     ICSA           OK    n= 3097  1967-01-07 -> 2026-05-09
nonfarm_payrolls   PAYEMS         OK    n= 1048  1939-01-01 -> 2026-04-01
payrolls_mom       PAYEMS         OK    n= 1048  1939-01-01 -> 2026-04-01
ig_spread          BAMLC0A0CM     OK    n=  792  2023-05-22 -> 2026-05-18
hy_spread          BAMLH0A0HYM2   OK    n=  792  2023-05-22 -> 2026-05-18
usd_index          DTWEXBGS       OK    n= 5315  2006-01-02 -> 2026-05-15


In [90]:
# Series to download: {fred_id: output_column_name}
SERIES = {
    # --- interest rates & yield curve ---
    "DGS10":       "yield_10y",          # 10-Year Treasury yield (daily)
    "DGS2":        "yield_2y",           # 2-Year Treasury yield (daily)
    "T10Y2Y":      "yield_curve_10y2y",  # 10Y-2Y spread (daily) — recession signal
    "T10Y3M":      "yield_curve_10y3m",  # 10Y-3M spread (daily) — classic inversion
    "FEDFUNDS":    "fed_funds_rate",      # Effective Fed Funds Rate (monthly)

    # --- credit & risk appetite ---
    "BAMLH0A0HYM2":  "hy_spread",        # High-yield OAS (daily) — risk-off signal
    "BAMLC0A0CM":    "ig_spread",         # Investment-grade OAS (daily)

    # --- volatility & sentiment ---
    "VIXCLS":      "vix",                # CBOE VIX (daily)

    # --- macro cycle ---
    "INDPRO":      "industrial_prod",    # Industrial Production Index (monthly)
    "UNRATE":      "unemployment",       # Unemployment rate (monthly)
    "CPIAUCSL":    "cpi",               # CPI All Urban (monthly)
    "PAYEMS":      "nonfarm_payrolls",   # Nonfarm payrolls (monthly)
    "ICSA":        "jobless_claims",     # Initial jobless claims (weekly)

    # --- commodities & dollar ---
    "DCOILWTICO":  "oil_wti",           # WTI crude oil (daily)
    "DTWEXBGS":    "usd_index",         # Trade-weighted USD (daily)
    "GOLDAMGBD228NLBM": "gold",         # Gold price (daily)
}

START = "2001-01-01"

raw = {}
for series_id, col in SERIES.items():
    try:
        raw[col] = fred.get_series(series_id, observation_start=START)
    except Exception as e:
        print(f"Failed {series_id}: {e}")

fred_df = pd.DataFrame(raw)
fred_df.index = pd.to_datetime(fred_df.index)
fred_df.index.name = "Date"

# Derived features worth having
fred_df["yield_spread_chg"] = fred_df["yield_curve_10y2y"].diff()
fred_df["cpi_yoy"]          = fred_df["cpi"].pct_change(12) * 100
fred_df["indpro_yoy"]       = fred_df["industrial_prod"].pct_change(12) * 100
fred_df["payrolls_mom"]     = fred_df["nonfarm_payrolls"].diff()

# Align to daily trading dates from the price data, forward-filling
# monthly/quarterly releases into each trading day
prices = pd.read_parquet("sp500_prices.parquet")
trading_days = pd.to_datetime(prices["Date"].unique())
trading_days = pd.DatetimeIndex(sorted(trading_days))

fred_daily = fred_df.reindex(trading_days, method="ffill")

fred_daily.to_parquet("fred_factors.parquet", index=True)
print(fred_daily.shape)
fred_daily.tail()

Failed GOLDAMGBD228NLBM: Bad Request.  The series does not exist.
(6382, 19)


,yield_10y,yield_2y,yield_curve_10y2y,yield_curve_10y3m,fed_funds_rate,hy_spread,ig_spread,vix,industrial_prod,unemployment,cpi,nonfarm_payrolls,jobless_claims,oil_wti,usd_index,yield_spread_chg,cpi_yoy,indpro_yoy,payrolls_mom
2026-05-13,4.46,3.98,0.48,0.77,NaN,2.82,0.76,17.87,NaN,NaN,NaN,NaN,NaN,NaN,118.4737,0.02,0.0,0.0,NaN
2026-05-14,4.47,4.00,0.47,0.78,NaN,2.76,0.76,17.26,NaN,NaN,NaN,NaN,NaN,NaN,118.6696,-0.01,0.0,0.0,NaN
2026-05-15,4.59,4.09,0.50,0.90,NaN,2.80,0.75,18.43,NaN,NaN,NaN,NaN,NaN,NaN,119.2825,0.03,0.0,0.0,NaN
2026-05-18,NaN,NaN,0.54,0.93,NaN,2.83,0.75,17.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.04,0.0,0.0,NaN
2026-05-19,NaN,NaN,0.54,0.93,NaN,2.83,0.75,17.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.04,0.0,0.0,NaN


In [91]:
SERIES = {
    # --- interest rates & yield curve ---
    "DGS10":       "yield_10y",          # 10-Year Treasury yield (daily)
    "DGS2":        "yield_2y",           # 2-Year Treasury yield (daily)
    "T10Y2Y":      "yield_curve_10y2y",  # 10Y-2Y spread (daily) — recession signal
    "T10Y3M":      "yield_curve_10y3m",  # 10Y-3M spread (daily) — classic inversion
    "FEDFUNDS":    "fed_funds_rate",      # Effective Fed Funds Rate (monthly)

    # --- credit & risk appetite ---
    "BAMLH0A0HYM2":  "hy_spread",        # High-yield OAS (daily) — risk-off signal
    "BAMLC0A0CM":    "ig_spread",         # Investment-grade OAS (daily)

    # --- volatility & sentiment ---
    "VIXCLS":      "vix",                # CBOE VIX (daily)

    # --- macro cycle ---
    "INDPRO":      "industrial_prod",    # Industrial Production Index (monthly)
    "UNRATE":      "unemployment",       # Unemployment rate (monthly)
    "CPIAUCSL":    "cpi",               # CPI All Urban (monthly)
    "PAYEMS":      "nonfarm_payrolls",   # Nonfarm payrolls (monthly)
    "ICSA":        "jobless_claims",     # Initial jobless claims (weekly)

    # --- commodities & dollar ---
    "DCOILWTICO":  "oil_wti",           # WTI crude oil (daily)
    "DTWEXBGS":    "usd_index",         # Trade-weighted USD (daily)
    "GOLDAMGBD228NLBM": "gold",         # Gold price (daily)
}

def load_fred_pit(series_id, daily_index, transform=None):
    """Fetch a FRED series as POINT-IN-TIME data.

    Each observation is timestamped at its FIRST-RELEASE date
    (realtime_start from ALFRED), so a backtest only ever sees a figure
    on/after the day it was actually published -- no lookahead, no
    fixed-lag guessing. Falls back to plain observation-date indexing
    if the series has no vintage history.

    Returns (aligned_series, is_pit); is_pit is False if it fell back.
    """
    try:
        raw = fred.get_series_all_releases(series_id)      # ALFRED vintages
    except Exception:
        raw = None

    if raw is not None and len(raw) > 0:
        # one row per observation = its FIRST release (earliest realtime_start)
        pit = (raw.sort_values("realtime_start")
                  .groupby("date", as_index=False).first()
                  .sort_values("date")
                  .reset_index(drop=True))
        pit["realtime_start"] = pd.to_datetime(pit["realtime_start"])

        # transform at NATIVE frequency, on observation-ordered values
        val = pit["value"].astype(float)
        if transform == "mom":
            val = val.diff()
        elif transform == "yoy":
            val = val.pct_change(12)

        # timestamp each value on its RELEASE date, not its observation date
        s = pd.Series(val.values, index=pit["realtime_start"].values)
        is_pit = True
    else:
        # no vintage history -> observation-date series (NOT point-in-time)
        s = fred.get_series(series_id)
        s.index = pd.to_datetime(s.index)
        if transform == "mom":
            s = s.diff()
        elif transform == "yoy":
            s = s.pct_change(12)
        is_pit = False

    # collapse same-day releases, then carry last known value onto trading days
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    return s.reindex(daily_index, method="ffill"), is_pit


# --------------------- build the point-in-time panel -----------------------
# the trading-day calendar = the unique dates in your price panel
daily_index = df.index.get_level_values("Date").unique().sort_values()          # your trading-day calendar

fred_pit = {}
for series_id, name in SERIES.items():
    try:
        col, is_pit = load_fred_pit(series_id, daily_index)
        fred_pit[name] = col
        tag = "PIT     " if is_pit else "FALLBACK"
        n  = col.notna().sum()
        fv = col.first_valid_index()
        print(f"{name:20s} {series_id:18s} {tag} non-null={n:5d}  "
              f"first={fv.date() if fv is not None else 'NaT'}")
    except Exception as e:
        print(f"{name:20s} {series_id:18s} FAIL  {type(e).__name__}: {e}")

fred_pit_df = pd.DataFrame(fred_pit, index=daily_index)
print(f"\nBuilt fred_pit_df: {fred_pit_df.shape[0]} rows x {fred_pit_df.shape[1]} cols")

yield_10y            DGS10              FALLBACK non-null= 6335  first=2001-01-02
yield_2y             DGS2               FALLBACK non-null= 6335  first=2001-01-02
yield_curve_10y2y    T10Y2Y             FALLBACK non-null= 6335  first=2001-01-02
yield_curve_10y3m    T10Y3M             FALLBACK non-null= 6335  first=2001-01-02
fed_funds_rate       FEDFUNDS           PIT      non-null= 6382  first=2001-01-02
hy_spread            BAMLH0A0HYM2       PIT      non-null=  751  first=2023-05-22
ig_spread            BAMLC0A0CM         PIT      non-null=  751  first=2023-05-22
vix                  VIXCLS             FALLBACK non-null= 6382  first=2001-01-02
industrial_prod      INDPRO             PIT      non-null= 6382  first=2001-01-02
unemployment         UNRATE             PIT      non-null= 6382  first=2001-01-02
cpi                  CPIAUCSL           PIT      non-null= 6382  first=2001-01-02
nonfarm_payrolls     PAYEMS             PIT      non-null= 6382  first=2001-01-02
jobless_claims  

In [92]:
fred_pit_df.to_parquet("fred_factors.parquet", index=True)

In [93]:
from edgar import *

set_identity("gsundaram1999@gmail.com")


In [94]:
tickers = sorted(df.index.get_level_values("Ticker").unique().tolist())
pd.Series(tickers, name="Ticker").to_csv("tickers_with_data.csv", index=False)
print(f"Saved {len(tickers)} tickers")

Saved 726 tickers


In [95]:
prices = pd.read_parquet("sp500_prices.parquet").set_index(["Date", "Ticker"]).sort_index()
fred   = pd.read_parquet("fred_factors.parquet")

# --- PIT shift ---
# FRED data on day D is assumed available only after market close on D.
# Shifting by 1 row means: the value that was at index D moves to index D+1,
# so the first trading session that can legally use a release is D+1.
fred_pit = fred.shift(1)

# --- join ---
# prices has (Date, Ticker) MultiIndex; fred_pit is indexed by Date alone.
# on="Date" tells pandas to match the Date level of prices' MultiIndex
# against fred_pit's index — every (Date, Ticker) row gets the same
# macro values for that date (already the prior day's release after shift).
df = prices.join(fred_pit, on="Date", how="left")

print(df.shape)
df.head()

(3747294, 20)


Open       High        Low      Close     Volume  \
Date       Ticker                                                          
2001-01-02 A       32.129270  32.129270  29.259251  30.340168    2261684   
           AAPL     0.222645   0.228257   0.217968   0.222645  452312000   
           ABT     11.261874  11.495887  11.232622  11.261874    5243919   
           ACGL     1.571624   1.571624   1.532003   1.532003      90000   
           ADBE    29.005323  29.067432  22.235344  23.221338   16813400   

                   yield_10y  yield_2y  yield_curve_10y2y  yield_curve_10y3m  \
Date       Ticker                                                              
2001-01-02 A             NaN       NaN                NaN                NaN   
           AAPL          NaN       NaN                NaN                NaN   
           ABT           NaN       NaN                NaN                NaN   
           ACGL          NaN       NaN                NaN                NaN   
           ADBE          NaN       NaN                NaN                NaN   

                   fed_funds_rate  hy_spread  ig_spread  vix  industrial_prod  \
Date       Ticker                                                               
2001-01-02 A                  NaN        NaN        NaN  NaN              NaN   
           AAPL               NaN        NaN        NaN  NaN              NaN   
           ABT                NaN        NaN        NaN  NaN              NaN   
           ACGL               NaN        NaN        NaN  NaN              NaN   
           ADBE               NaN        NaN        NaN  NaN              NaN   

                   unemployment  cpi  nonfarm_payrolls  jobless_claims  \
Date       Ticker                                                        
2001-01-02 A                NaN  NaN               NaN             NaN   
           AAPL             NaN  NaN               NaN             NaN   
           ABT              NaN  NaN               NaN             NaN   
           ACGL             NaN  NaN               NaN             NaN   
           ADBE             NaN  NaN               NaN             NaN   

                   oil_wti  usd_index  
Date       Ticker                      
2001-01-02 A           NaN        NaN  
           AAPL        NaN        NaN  
           ABT         NaN        NaN  
           ACGL        NaN        NaN  
           ADBE        NaN        NaN

In [96]:
price_cols = ["Open", "High", "Low", "Close", "Volume"]

# --- overall null counts ---
print("=== Null counts per price column ===")
print(df[price_cols].isnull().sum())
print()

# --- per-ticker row count and null summary ---
ticker_stats = (
    df[price_cols]
    .groupby(level="Ticker")
    .agg(
        rows        = ("Close", "count"),        # non-null close rows
        total_rows  = ("Close", "size"),         # total rows incl. NaN
        first_date  = ("Close", lambda s: s.first_valid_index()[0] if s.first_valid_index() else None),
        last_date   = ("Close", lambda s: s.last_valid_index()[0]  if s.last_valid_index()  else None),
        null_close  = ("Close", lambda s: s.isnull().sum()),
    )
)
ticker_stats["pct_missing"] = (ticker_stats["null_close"] / ticker_stats["total_rows"] * 100).round(2)

print("=== Per-ticker coverage ===")
print(ticker_stats.sort_values("pct_missing", ascending=False).to_string())
print()
print(f"Tickers with >5% missing Close: {(ticker_stats['pct_missing'] > 5).sum()}")
print(f"Tickers with any missing Close: {(ticker_stats['null_close'] > 0).sum()}")

=== Null counts per price column ===
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

=== Per-ticker coverage ===
        rows  total_rows first_date  last_date  null_close  pct_missing
Ticker                                                                 
A       6382        6382 2001-01-02 2026-05-19           0          0.0
NOC     6382        6382 2001-01-02 2026-05-19           0          0.0
NOW     3491        3491 2012-06-29 2026-05-19           0          0.0
NRG     5651        5651 2003-12-02 2026-05-19           0          0.0
NSC     6382        6382 2001-01-02 2026-05-19           0          0.0
NSM     1610        1610 2012-03-08 2018-07-31           0          0.0
NTAP    6382        6382 2001-01-02 2026-05-19           0          0.0
NTRS    6382        6382 2001-01-02 2026-05-19           0          0.0
NUE     6382        6382 2001-01-02 2026-05-19           0          0.0
NVDA    6382        6382 2001-01-02 2026-05-19           0          0

In [97]:
fred_cols  = [c for c in df.columns if c not in price_cols]

# --- collapse to one row per date (FRED is date-level, not ticker-level) ---
fred_daily_check = df[fred_cols].groupby(level="Date").first()

print(f"Date range: {fred_daily_check.index.min().date()} → {fred_daily_check.index.max().date()}")
print(f"Total trading days: {len(fred_daily_check)}")
print()

# --- null counts and % missing per series ---
stats = pd.DataFrame({
    "non_null":    fred_daily_check.notna().sum(),
    "null":        fred_daily_check.isnull().sum(),
    "pct_missing": (fred_daily_check.isnull().sum() / len(fred_daily_check) * 100).round(2),
    "first_valid": fred_daily_check.apply(lambda s: s.first_valid_index()),
    "last_valid":  fred_daily_check.apply(lambda s: s.last_valid_index()),
})

print("=== FRED series coverage ===")
print(stats.sort_values("pct_missing", ascending=False).to_string())

Date range: 2001-01-02 → 2026-05-19
Total trading days: 6382

=== FRED series coverage ===
                   non_null  null  pct_missing first_valid last_valid
hy_spread               750  5632        88.25  2023-05-23 2026-05-19
ig_spread               750  5632        88.25  2023-05-23 2026-05-19
usd_index              1832  4550        71.29  2019-02-05 2026-05-19
oil_wti                3773  2609        40.88  2011-04-07 2026-05-19
jobless_claims         4270  2112        33.09  2009-05-29 2026-05-19
yield_10y              6334    48         0.75  2001-01-03 2026-05-19
yield_2y               6334    48         0.75  2001-01-03 2026-05-19
yield_curve_10y2y      6334    48         0.75  2001-01-03 2026-05-19
yield_curve_10y3m      6334    48         0.75  2001-01-03 2026-05-19
fed_funds_rate         6381     1         0.02  2001-01-03 2026-05-19
vix                    6381     1         0.02  2001-01-03 2026-05-19
industrial_prod        6381     1         0.02  2001-01-03 2026-05-19

In [98]:
# --- filter to 2005+ ---
df = df[df.index.get_level_values("Date") >= "2005-01-01"]

# --- split column groups ---
price_cols = ["Open", "High", "Low", "Close", "Volume"]
fred_cols  = [c for c in df.columns if c not in price_cols]

# --- fill ---
# FRED: ffill within each column across dates (monthly releases carry forward)
# Price: bfill within each ticker (rare gaps filled from the next available day)
df[fred_cols]  = df[fred_cols].groupby(level="Ticker").ffill()
df[price_cols] = df[price_cols].groupby(level="Ticker").bfill()

print(df.shape)
print(df.isnull().sum())

(3260173, 20)
Open                       0
High                       0
Low                        0
Close                      0
Volume                     0
yield_10y                  0
yield_2y                   0
yield_curve_10y2y          0
yield_curve_10y3m          0
fed_funds_rate             0
hy_spread            2762973
ig_spread            2762973
vix                        0
industrial_prod            0
unemployment               0
cpi                        0
nonfarm_payrolls           0
jobless_claims        590072
oil_wti               854690
usd_index            2067464
dtype: int64


In [99]:
df

Open        High         Low       Close     Volume  \
Date       Ticker                                                              
2005-01-03 A        14.372443   14.420152   14.014623   14.241242    3378826   
           AAP      24.492349   24.625945   24.241859   24.241859    1549500   
           AAPL      0.969609    0.974548    0.936979    0.947307  691992000   
           ABT      13.617552   13.711264   13.567767   13.667336    7165902   
           ACGL      4.067733    4.078298    4.014905    4.025470    1130400   
...                       ...         ...         ...         ...        ...   
2026-05-19 YUM     152.009995  153.695007  150.669998  153.335007     650188   
           ZBH      85.250000   86.364998   84.290001   86.089996     923849   
           ZBRA    259.640015  259.845001  249.699997  250.520004     477407   
           ZION     60.130001   60.470001   59.189999   60.310001     367297   
           ZTS      79.849998   80.360001   77.593002   77.889999    3895872   

                   yield_10y  yield_2y  yield_curve_10y2y  yield_curve_10y3m  \
Date       Ticker                                                              
2005-01-03 A            4.24      3.08               1.16               2.02   
           AAP          4.24      3.08               1.16               2.02   
           AAPL         4.24      3.08               1.16               2.02   
           ABT          4.24      3.08               1.16               2.02   
           ACGL         4.24      3.08               1.16               2.02   
...                      ...       ...                ...                ...   
2026-05-19 YUM          4.59      4.09               0.54               0.93   
           ZBH          4.59      4.09               0.54               0.93   
           ZBRA         4.59      4.09               0.54               0.93   
           ZION         4.59      4.09               0.54               0.93   
           ZTS          4.59      4.09               0.54               0.93   

                   fed_funds_rate  hy_spread  ig_spread    vix  \
Date       Ticker                                                
2005-01-03 A                 1.93        NaN        NaN  13.29   
           AAP               1.93        NaN        NaN  13.29   
           AAPL              1.93        NaN        NaN  13.29   
           ABT               1.93        NaN        NaN  13.29   
           ACGL              1.93        NaN        NaN  13.29   
...                           ...        ...        ...    ...   
2026-05-19 YUM               3.64       2.83       0.75  17.82   
           ZBH               3.64       2.83       0.75  17.82   
           ZBRA              3.64       2.83       0.75  17.82   
           ZION              3.64       2.83       0.75  17.82   
           ZTS               3.64       2.83       0.75  17.82   

                   industrial_prod  unemployment      cpi  nonfarm_payrolls  \
Date       Ticker                                                             
2005-01-03 A              117.6000           5.4  191.200          132075.0   
           AAP            117.6000           5.4  191.200          132075.0   
           AAPL           117.6000           5.4  191.200          132075.0   
           ABT            117.6000           5.4  191.200          132075.0   
           ACGL           117.6000           5.4  191.200          132075.0   
...                            ...           ...      ...               ...   
2026-05-19 YUM            102.4963           4.3  332.407          158736.0   
           ZBH            102.4963           4.3  332.407          158736.0   
           ZBRA           102.4963           4.3  332.407          158736.0   
           ZION           102.4963           4.3  332.407          158736.0   
           ZTS            102.4963           4.3  332.407          158736.0   

                   jobless_claims  oil_wti  usd_index  


In [100]:
EMA_WINDOW = 20  # modifiable — span in trading sessions

# --- forward returns (within each ticker to avoid cross-ticker bleed) ---
df["fwd_ret_5"] = df.groupby(level="Ticker")["Close"].transform(
    lambda s: s.shift(-5) / s - 1
)

# --- valid FRED cols: drop anything missing more than 10% of dates ---
fred_daily = df[fred_cols].groupby(level="Date").first()
fred_daily = df[fred_cols].groupby(level="Date").first()

# keep only numeric columns — datetime cols (e.g. first_valid_index artefacts) break ewm
fred_daily_numeric = fred_daily.select_dtypes(include="number")

valid_fred_cols = fred_daily_numeric.columns[
    fred_daily_numeric.isnull().mean() < 0.10
].tolist()
print(f"Valid FRED cols ({len(valid_fred_cols)}): {valid_fred_cols}")

ema_dev_cols = []
for col in valid_fred_cols:
    dev_col = f"{col}_ema{EMA_WINDOW}_dev"
    fred_daily_numeric[dev_col] = (
        fred_daily_numeric[col]
        - fred_daily_numeric[col].ewm(span=EMA_WINDOW, min_periods=EMA_WINDOW).mean()
    )
    ema_dev_cols.append(dev_col)

for dev_col in ema_dev_cols:
    df[dev_col] = df.index.get_level_values("Date").map(fred_daily_numeric[dev_col])

print(f"\nForward return non-null rows : {df['fwd_ret_5'].notna().sum():,}")
print(f"EMA deviation non-null rows  : {df[ema_dev_cols].notna().all(axis=1).sum():,}")
print(f"\nSample of new columns:")
df[["fwd_ret_5"] + ema_dev_cols].dropna().head()

Valid FRED cols (10): ['yield_10y', 'yield_2y', 'yield_curve_10y2y', 'yield_curve_10y3m', 'fed_funds_rate', 'vix', 'industrial_prod', 'unemployment', 'cpi', 'nonfarm_payrolls']

Forward return non-null rows : 3,256,547
EMA deviation non-null rows  : 3,250,561

Sample of new columns:


fwd_ret_5  yield_10y_ema20_dev  yield_2y_ema20_dev  \
Date       Ticker                                                       
2005-01-31 A        0.045681            -0.047592            0.022274   
           AAP      0.046868            -0.047592            0.022274   
           AAPL     0.026528            -0.047592            0.022274   
           ABT      0.012661            -0.047592            0.022274   
           ACGL     0.078955            -0.047592            0.022274   

                   yield_curve_10y2y_ema20_dev  yield_curve_10y3m_ema20_dev  \
Date       Ticker                                                             
2005-01-31 A                         -0.069866                    -0.127455   
           AAP                       -0.069866                    -0.127455   
           AAPL                      -0.069866                    -0.127455   
           ABT                       -0.069866                    -0.127455   
           ACGL                      -0.069866                    -0.127455   

                   fed_funds_rate_ema20_dev  vix_ema20_dev  \
Date       Ticker                                            
2005-01-31 A                       0.003782      -0.243667   
           AAP                     0.003782      -0.243667   
           AAPL                    0.003782      -0.243667   
           ABT                     0.003782      -0.243667   
           ACGL                    0.003782      -0.243667   

                   industrial_prod_ema20_dev  unemployment_ema20_dev  \
Date       Ticker                                                      
2005-01-31 A                         0.05725                     0.0   
           AAP                       0.05725                     0.0   
           AAPL                      0.05725                     0.0   
           ABT                       0.05725                     0.0   
           ACGL                      0.05725                     0.0   

                   cpi_ema20_dev  nonfarm_payrolls_ema20_dev  
Date       Ticker                                             
2005-01-31 A           -0.036296                   19.376574  
           AAP         -0.036296                   19.376574  
           AAPL        -0.036296                   19.376574  
           ABT         -0.036296                   19.376574  
           ACGL        -0.036296                   19.376574

In [101]:
df = df.copy()

# Regression on Unwinsorized data giving near 0 R^2

### Below, we see that due to fat tails, the regression model does not work

In [102]:
import statsmodels.api as sm

# --- build regression dataset ---
reg_cols = ["fwd_ret_5"] + ema_dev_cols
reg_df   = df[reg_cols].dropna()

print(f"Regression rows: {len(reg_df):,}  |  Features: {len(ema_dev_cols)}")

y = reg_df["fwd_ret_5"]
X = sm.add_constant(reg_df[ema_dev_cols])

# --- OLS ---
# Use HC3 robust SEs — observations are not independent (same macro value
# appears across all tickers on a given date), so standard SEs are too small.
model  = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())

# --- compact view: only significant predictors ---
results_df = pd.DataFrame({
    "coef"   : model.params,
    "pvalue" : model.pvalues,
    "tstat"  : model.tvalues,
}).drop("const").sort_values("pvalue")

print("\n=== Predictors sorted by p-value ===")
print(results_df.to_string())
print(f"\nR²: {model.rsquared:.6f}   Adj-R²: {model.rsquared_adj:.6f}")

Regression rows: 3,246,935  |  Features: 10
                            OLS Regression Results                            
Dep. Variable:              fwd_ret_5   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     5.232
Date:                Tue, 19 May 2026   Prob (F-statistic):           3.78e-07
Time:                        19:18:43   Log-Likelihood:            -2.0930e+07
No. Observations:             3246935   AIC:                         4.186e+07
Df Residuals:                 3246925   BIC:                         4.186e+07
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

/Users/pasqualebifulco/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 10, but rank is 9
  warnings.warn('covariance of constraints does not have full '


In [103]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ----------------------------------------------------------------------
# Drop-in replacement.
# Why: all 15 features are macro -> identical across tickers on a given
# date. The 3.1M-row panel is the same ~few-hundred macro observations
# repeated ~500x. The repetition adds no information; it just shrinks the
# standard errors to fiction. This collapses the panel to one row per
# date -- the time-series regression it actually is -- and uses
# Newey-West SEs to handle the autocorrelation from overlapping 5-day
# forward returns.
# ----------------------------------------------------------------------
df["date"] = df.index.get_level_values("Date")
DATE_COL    = "date"    # <-- set to your date column (or index name)
RET_OVERLAP = 5         # fwd_ret_5 overlaps 5 days -> Newey-West lag floor

reg_cols = ["fwd_ret_5"] + ema_dev_cols

# get date as a column whether it's currently the index or a column
work = df.reset_index() if DATE_COL not in df.columns else df.copy()

# tame fat tails in raw 5-day returns before aggregating
lo, hi = work["fwd_ret_5"].quantile([0.001, 0.995])
work["fwd_ret_5"] = work["fwd_ret_5"].clip(lo, hi)

# collapse panel -> one row per date.
# fwd_ret_5 becomes the equal-weighted cross-sectional mean return;
# the macro features are constant per date, so .mean() just recovers them.
daily = (work[[DATE_COL] + reg_cols]
         .dropna()
         .groupby(DATE_COL)
         .mean()
         .sort_index())

print(f"Panel rows: {len(work):,}  ->  daily rows: {len(daily):,}  "
      f"(this is your real sample size)")

y = daily["fwd_ret_5"]
X = sm.add_constant(daily[ema_dev_cols])

# Newey-West / HAC SEs: lag >= the return overlap to absorb the
# autocorrelation induced by overlapping 5-day windows.
maxlags = max(RET_OVERLAP, int(round(1.5 * RET_OVERLAP)))
model   = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
print(model.summary())

# --- collinearity check: the yield/curve features are near-redundant ---
vif = pd.DataFrame({
    "feature": ema_dev_cols,
    "VIF": [variance_inflation_factor(X.values, i + 1)   # +1 skips const
            for i in range(len(ema_dev_cols))],
}).sort_values("VIF", ascending=False)
print("\n=== VIF (>~10 = coefficient not interpretable; inf = drop it) ===")
print(vif.to_string(index=False))

# --- predictors sorted by p-value ---
results_df = (pd.DataFrame({
        "coef"  : model.params,
        "pvalue": model.pvalues,
        "tstat" : model.tvalues,
    })
    .drop("const")
    .sort_values("pvalue"))
print("\n=== Predictors sorted by p-value (HAC SEs) ===")
print(results_df.to_string())
print(f"\nR²: {model.rsquared:.6f}   Adj-R²: {model.rsquared_adj:.6f}")
print(f"N (days): {int(model.nobs)}   HAC maxlags: {maxlags}")

Panel rows: 3,260,173  ->  daily rows: 5,354  (this is your real sample size)
                            OLS Regression Results                            
Dep. Variable:              fwd_ret_5   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     2.889
Date:                Tue, 19 May 2026   Prob (F-statistic):            0.00208
Time:                        19:18:44   Log-Likelihood:                 11770.
No. Observations:                5354   AIC:                        -2.352e+04
Df Residuals:                    5344   BIC:                        -2.345e+04
Df Model:                           9                                         
Covariance Type:                  HAC                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------

/Users/pasqualebifulco/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 10, but rank is 9
  warnings.warn('covariance of constraints does not have full '
/Users/pasqualebifulco/anaconda3/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


Directional accuracy + signal/return correlation.

`signal` = the model's linear prediction, DEMEANED: the intercept is
constant and carries zero timing information, so |signal| is only
meaningful relative to the signal's own mean. Both series are demeaned,
so 50% directional accuracy = no skill.

In [104]:
import numpy as np

signal = model.fittedvalues - model.fittedvalues.mean()   # demeaned prediction
actual = y - y.mean()                                      # demeaned return
sd_sig = signal.std()

# correlation of realized return with signal (invariant to demeaning).
# for an IN-SAMPLE OLS fit this is mechanically sqrt(R^2) -- redundant here;
# it only becomes independent information for OOS predictions.
corr = np.corrcoef(y, model.fittedvalues)[0, 1]
print(f"corr(actual, signal): {corr:+.4f}   "
      f"(in-sample = sqrt(R^2) = {np.sqrt(model.rsquared):.4f})\n")

# directional accuracy at increasing signal-strength thresholds.
# 50% = coin flip. A real signal -> dir_acc rises as the threshold rises.
hdr = f"{'threshold':>12}{'n_obs':>9}{'kept':>8}{'dir_acc':>10}{'+-SE':>8}"
print(hdr); print('-' * len(hdr))
for n in (0, 1, 2, 3):
    mask = signal.abs() > n * sd_sig
    k = int(mask.sum())
    if k == 0:
        print(f"{f'|sig|>{n}sd':>12}{0:>9}{'--':>8}{'--':>10}{'--':>8}")
        continue
    hit = (np.sign(signal[mask]) == np.sign(actual[mask])).mean()
    se  = np.sqrt(hit * (1 - hit) / k)        # binomial SE -- see caveat below
    print(f"{f'|sig|>{n}sd':>12}{k:>9}{k/len(signal)*100:>7.1f}%"
          f"{hit*100:>9.1f}%{se*100:>7.1f}%")

corr(actual, signal): +0.1355   (in-sample = sqrt(R^2) = 0.1355)

   threshold    n_obs    kept   dir_acc    +-SE
-----------------------------------------------
   |sig|>0sd     5354  100.0%     50.7%    0.7%
   |sig|>1sd      829   15.5%     56.6%    1.7%
   |sig|>2sd      181    3.4%     68.5%    3.5%
   |sig|>3sd       80    1.5%     77.5%    4.7%


In [105]:
import numpy as np
from pathlib import Path

# ─── Load & featurize Form 4 data ────────────────────────────────────────────

# 1. Load all available CSVs
form4_files = sorted(Path("form4_data").glob("form4_*.csv"))
print(f"Loading {len(form4_files)} Form 4 files...")
form4_raw = pd.concat([pd.read_csv(f) for f in form4_files], ignore_index=True)
print(f"Total raw transactions: {len(form4_raw):,}")

# 2. Filter to open-market buys (P) and sales (S) only
#    Exclude: M (option exercise), F (tax withholding), G (gift), A (grant)
form4_raw["filing_date"] = pd.to_datetime(form4_raw["filing_date"]).dt.normalize()
form4_raw["Shares"] = pd.to_numeric(form4_raw["Shares"], errors="coerce")
form4_raw["Value"]  = pd.to_numeric(form4_raw["Value"],  errors="coerce")

open_mkt = form4_raw[form4_raw["Code"].isin(["P", "S"])].copy()
sign = open_mkt["Code"].map({"P": 1, "S": -1})
open_mkt["signed_shares"] = open_mkt["Shares"] * sign
open_mkt["signed_value"]  = open_mkt["Value"]  * sign

# 3. Aggregate to (filing_date, ticker): net buys/sells per filing day
f4_daily = (
    open_mkt
    .groupby(["filing_date", "query_ticker"])
    .agg(
        net_shares=("signed_shares", "sum"),
        net_value =("signed_value",  "sum"),
        buy_count =("Code", lambda x: (x == "P").sum()),
        sell_count=("Code", lambda x: (x == "S").sum()),
    )
    .reset_index()
    .rename(columns={"query_ticker": "Ticker"})
)
print(f"Filing events: {len(f4_daily):,}  |  Tickers: {f4_daily['Ticker'].nunique()}")

# 4. PIT shift: filing on day D → available next trading session (D+1)
trading_dates = df.index.get_level_values("Date").unique().sort_values()
td_arr = trading_dates.values
filing_arr = f4_daily["filing_date"].values
idx = np.searchsorted(td_arr, filing_arr, side="right")   # first date strictly after filing
valid = idx < len(td_arr)
f4_daily = f4_daily[valid].copy()
f4_daily["avail_date"] = pd.DatetimeIndex(td_arr[idx[valid]])

# 5. Set MultiIndex, collapse same (avail_date, ticker) duplicates
f4_cols = ["net_shares", "net_value", "buy_count", "sell_count"]
f4_mi = (
    f4_daily.set_index(["avail_date", "Ticker"])[f4_cols]
    .rename_axis(["Date", "Ticker"])
    .groupby(level=["Date", "Ticker"]).sum()
)

# 6. Expand to full (trading_dates × form4 tickers) grid, 0 on silent days
f4_tickers = f4_mi.index.get_level_values("Ticker").unique()
full_idx   = pd.MultiIndex.from_product([trading_dates, f4_tickers], names=["Date", "Ticker"])
f4_full    = f4_mi.reindex(full_idx, fill_value=0).sort_index()

# 7. Rolling 20-session cumulative sum per ticker
#    Unstack → rolling on date axis (naturally per-ticker column) → stack back
ROLL_WINDOW = 20
f4_wide   = f4_full.unstack("Ticker")
roll_wide = f4_wide.rolling(window=ROLL_WINDOW, min_periods=1).sum()
roll_f4   = roll_wide.stack("Ticker")[f4_cols].sort_index()
roll_f4.columns = [f"insider_{c}_roll{ROLL_WINDOW}" for c in f4_cols]
roll_f4.index.names = ["Date", "Ticker"]

print(f"Insider feature grid: {roll_f4.shape}  (dates × tickers with form4 data)")

# 8. Join to main panel (NaN for tickers with no form4 data)
for col in roll_f4.columns:
    df[col] = roll_f4[col]

print(f"Panel shape after Form 4 join: {df.shape}")
df[[c for c in df.columns if c.startswith("insider_")]].describe()

Loading 1 Form 4 files...
Total raw transactions: 1
Filing events: 1  |  Tickers: 1
Insider feature grid: (5378, 4)  (dates × tickers with form4 data)
Panel shape after Form 4 join: (3260173, 36)


,insider_net_shares_roll20,insider_net_value_roll20,insider_buy_count_roll20,insider_sell_count_roll20
count,5378.0,5378.0,5378.000000,5378.0
mean,0.0,0.0,0.003719,0.0
std,0.0,0.0,0.060875,0.0
min,0.0,0.0,0.000000,0.0
25%,0.0,0.0,0.000000,0.0
50%,0.0,0.0,0.000000,0.0
75%,0.0,0.0,0.000000,0.0
max,0.0,0.0,1.000000,0.0
